# B2-024 — Practice p07

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** gpu-colab-l4-workflow

## Task

Implement checkpoints that make a resumed run reproduce the uninterrupted run exactly.

**Contract.**

* `save_checkpoint(path, model, optimizer, step)` writes, with `torch.save`, a dict with exactly the keys `"model"` (`model.state_dict()`), `"optimizer"` (`optimizer.state_dict()`), `"step"` (an `int`), `"torch_rng"` (`torch.get_rng_state()`), and `"cuda_rng"` (`torch.cuda.get_rng_state_all()` when CUDA is available, otherwise `None`).
* `load_checkpoint(path, model, optimizer)` reads it with `torch.load(path, weights_only=True)`, loads both state dicts, restores the global RNG with `torch.set_rng_state` (and the CUDA states with `torch.cuda.set_rng_state_all` when present and available), and returns the step.

**Training loop for the probe.** Load the unit loader with the supplied cell and use `x, y = capstone_data.train_rows("shapes_supervised")`.
Write `train_steps(model, optimizer, n_steps)` that performs `n_steps` updates, each drawing `idx = torch.randint(0, 600, (64,))` from the **global** generator and then doing one update on `(x[idx], y[idx])` with `F.cross_entropy`; it returns the list of loss tensors (detached). Each update runs, in this order: `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()`; no scheduler, no gradient clipping, no accumulation, no early stopping.
A *fresh run* means `torch.manual_seed(SEED)`, then `model = capstone_data.ShapeCNN()`, then `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`.

**Probes.** Use a temporary directory (`tempfile.TemporaryDirectory()`).

1. *Uninterrupted:* fresh run, `train_steps(..., 21)`; keep `L21`, the 21st loss.
2. *Resumed:* fresh run, `train_steps(..., 20)`, `save_checkpoint(path, model, optimizer, 20)`. Then simulate a new session: fresh run again (new model and optimizer), then `torch.manual_seed(SEED + 1)` to move the generator elsewhere, then `load_checkpoint`. It must return `20`, and one more `train_steps(..., 1)` must give a loss equal to `L21` under `torch.equal`. After loading, every tensor of the model's `state_dict` equals the saved one (`torch.equal`).
3. *Wrong variant:* repeat probe 2 with a local copy of `load_checkpoint` that skips the RNG restore; its next loss must **not** be `torch.equal` to `L21`. Explain in two sentences why it differs and why it still trains normally.
4. The saved file's keys are exactly the five listed above, and `"cuda_rng"` is `None` on a CPU-only machine.

**Required answer-check assertions.** Probes 1–4, including both `torch.equal` outcomes and the returned step.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import os
import tempfile

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, the same `save_checkpoint` / `load_checkpoint` protect a long run against disconnects. Use a replacement config for the same training code (move each batch with `.to(device)`), and write checkpoints to mounted persistent storage with an atomic rename (`torch.save` to `path + ".tmp"`, then `os.replace`).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "batch_size": 512, "steps": 20000,
             "checkpoint_every": 2000, "checkpoint_dir": "/content/drive/MyDrive/b2-024-checkpoints"}
```

**What to observe.** `"cuda_rng"` is now a non-empty list; a resume after a forced runtime restart continues from the last saved step; with CUDA kernels the next-step loss may differ from the uninterrupted run in the last bits unless deterministic algorithms are enabled, so compare with a tolerance there and keep `torch.equal` for the CPU certificate.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.